In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import os
os.environ["PYTHONUTF8"] = "1"
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)



In [ ]:
# load file
DATA_FILE = 'hypotension_patients.csv'
#Exploratory Data Analysis
print("Step 1: Exploratory Data Analysis")

# Load the dataset
try:
    df = pd.read_csv(DATA_FILE)
except FileNotFoundError:
    print(f"Error: The data file '{DATA_FILE}' was not found.")


#Data Preprocessing
# Create int variable 'in_hospital_mortality', remove non-number
df['in_hospital_mortality'] = df['dod'].notna().astype(int)

#feature lists for visualizations
numerical_features = ['anchor_age', 'apsiii', 'charlson_comorbidity_index']
categorical_features = ['gender']
target = 'in_hospital_mortality'


# table
print("\nSummary for Numerical Features:")
summary_stats = df[numerical_features].describe().loc[['mean', 'std']]
print(summary_stats)


# Histograms
plt.figure(figsize=(15, 5))
for i, feature in enumerate(numerical_features):
    #diagram groups in the same row and 3 columns
    plt.subplot(1, 3, i + 1)
    #discrete=True to avoid gaps in the histogram
    sns.histplot(df[feature], kde=True, bins=30, discrete=True)
    #3 numerical features
    plt.title(f'Distribution of {feature.replace("_", " ").title()}', fontsize=14)
    plt.xlabel(feature.replace("_", " ").title())
    plt.ylabel('Frequency')

plt.suptitle('Histograms of Numerical Features', fontsize=18, y=1.03)
#adjust 3 diagrams' gaps automatically
plt.tight_layout()
plt.show()

print('\n')
#Categorical and Target Variables
plt.figure(figsize=(10, 5))
# Gender Distribution
plt.subplot(1, 2, 1)
sns.countplot(x='gender', data=df, palette='pastel')
plt.title('Gender Distribution', fontsize=14)
plt.xlabel('Gender')
plt.ylabel('Count')
# In-Hospital Mortality Distribution
plt.subplot(1, 2, 2)
sns.countplot(x=target, data=df, palette='pastel')
plt.title('In-Hospital Mortality Distribution', fontsize=14)
plt.xlabel('Result (0: Alive, 1: Died)')
plt.ylabel('Count')

plt.suptitle('Distributions of Categorical and Target Variables', fontsize=18, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Linear MLP Development and Hyperparameter Tuning with scikit-learn
print("\nStep 2: Linear MLP Development ")

#Data preparation
#independent var. for learning
X = df[numerical_features + categorical_features]
#dependent var. for prediction
y = df[target]
#20% for test and 80 for train, stratify=y makes sure the split ratio is the same for multi-layer sampling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nData Split:")
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# preprocessing pipelines
numerical_transformer = Pipeline(steps=[
    #fill the empty with mean value
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    #one hot binary encoding, if seeing something in test but not train set then ignore the value
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
#combination
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# adam optimizer to train at most 500 times,15%data is for validation, if performance is not increased after 10 times then stop early
mlp = MLPClassifier(activation='identity', solver='adam', max_iter=500,
                    early_stopping=True, validation_fraction=0.15,
                    n_iter_no_change=10, random_state=42)

# Define hyperparameter grid as the instructions require
#layer with 1 2 3 and neurons per layer 16 32 64
n_layers_opts = [1, 2, 3]
n_neurons_opts = [16, 32, 64]
param_grid = {
    'classifier__hidden_layer_sizes': [(neurons,) * layers for layers in n_layers_opts for neurons in n_neurons_opts],
    #learning rate
    'classifier__learning_rate_init': [1e-2, 1e-3, 1e-4],
}

model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('classifier', mlp)])
#Hyperparameter Tuning with GridSearchCV for the best coefficient for each fold, 5 folds totally
grid_search = GridSearchCV(
    model_pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=1, verbose=0
)
grid_search.fit(X_train, y_train)

print("\nBest Parameters Found for Linear Model:")
# Get the best parameters
best_params = grid_search.best_params_

#a copy to format for display
display_params = best_params.copy()

#Check if hidden_layer_sizes is a tuple with a single element
hidden_layer_config = display_params['classifier__hidden_layer_sizes']
if isinstance(hidden_layer_config, tuple) and len(hidden_layer_config) == 1:
    #if so, replace the tuple with just the integer for display
    display_params['classifier__hidden_layer_sizes'] = hidden_layer_config[0]

print(display_params)

results_df = pd.DataFrame(grid_search.cv_results_)
# Process results for plotting
# the number of hidden layer
results_df['layers'] = results_df['param_classifier__hidden_layer_sizes'].apply(lambda x: len(x))
#get the #neurons in the first hidden layer
results_df['neurons'] = results_df['param_classifier__hidden_layer_sizes'].apply(lambda x: x[0] if x else 0)
learning_rates = results_df['param_classifier__learning_rate_init'].unique()

#drawing preparation
fig, axes = plt.subplots(1, len(learning_rates), figsize=(20, 6), sharey=True)

for i, lr in enumerate(learning_rates):
    subset = results_df[results_df['param_classifier__learning_rate_init'] == lr]
    pivot = subset.pivot(index='layers', columns='neurons', values='mean_test_score')
    #seaborn heatmap:show value with 4 digits; the deeper color the higher score, all maps have the same range
    sns.heatmap(pivot, ax=axes[i], annot=True, fmt=".4f", cmap="inferno",
                vmin=results_df['mean_test_score'].min(), vmax=results_df['mean_test_score'].max())
    axes[i].set_title(f'Mean CV Accuracy (LR = {lr})')
    axes[i].set_xlabel('# of Neurons per Layer')
    axes[i].set_ylabel('# of Hidden Layers')

plt.suptitle('Model Performance by Hyperparameters Tuning', fontsize=20, y=1.03)
plt.tight_layout()


In [ ]:
print("\nStep 3: Evaluating Non-Linearity")

#Best Linear Model Evaluation
best_linear_model = grid_search.best_estimator_
y_pred_linear = best_linear_model.predict(X_test)

#Non-Linear one with ReLU
#get the best parameters from the linear model search
best_hidden_layers = grid_search.best_params_['classifier__hidden_layer_sizes']
best_lr = grid_search.best_params_['classifier__learning_rate_init']

#adam optimizer with ReLU activation to train at most 500 times,15%data is for validation, if performance is not increased after 10 times then stop early
relu_mlp = MLPClassifier(activation='relu', hidden_layer_sizes=best_hidden_layers,
                         learning_rate_init=best_lr, solver='adam', max_iter=500,
                         early_stopping=True, validation_fraction=0.15,
                         n_iter_no_change=10, random_state=42)

#train the non-linear pipeline just like we did for linear model
nonlinear_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                     ('classifier', relu_mlp)])
nonlinear_pipeline.fit(X_train, y_train)
y_pred_nonlinear = nonlinear_pipeline.predict(X_test)


#Performance comparison
def calculate_metrics(y_true, y_pred):
    """Calculates all relevant metrics and returns them in a dictionary."""
    return {
        ##we gonna compare these 4 metrics
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred)
    }
##get the results for both
linear_metrics = calculate_metrics(y_test, y_pred_linear)
nonlinear_metrics = calculate_metrics(y_test, y_pred_nonlinear)

# display it as a table
metrics_table = pd.DataFrame({
    'Linear Model': linear_metrics,
    'Non-Linear Model (ReLU)': nonlinear_metrics
})
print("\nModel Performance Comparison")
print(metrics_table.round(4))

#preparation for confusion matrix
cm_linear = confusion_matrix(y_test, y_pred_linear)
cm_nonlinear = confusion_matrix(y_test, y_pred_nonlinear)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
#2 types
class_names = ['Alive', 'Died']

# heatmap in linear CM, all values should be integer
sns.heatmap(cm_linear, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('Linear Model Confusion Matrix', fontsize=16)
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

#heatmap in non-linear CM
sns.heatmap(cm_nonlinear, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title('Non-Linear Model (ReLU) Confusion Matrix', fontsize=16)
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.suptitle('Model Performance Comparison on Test Set', fontsize=20, y=1.03)
plt.tight_layout()
plt.show()